## Embedding based retrieval: finding the right story by meaning

Search engines used to work on words. You typed `landslide highway`. The engine
looked for documents that contained the string `landslide` and the string
`highway`, then ranked them by how often those strings appeared.

That works well until somebody asks the question in their own words.

> which road got buried by mud?

Not one of those three words appears in the article. A word matching engine
returns nothing at all.

In this notebook we build the other kind of search. A frozen encoder turns every
sentence into a point in space, and it does this so that sentences with a
similar **meaning** land near each other.

$$
\text{"the road was buried by mud"} \;\xrightarrow{\;\text{encoder}\;}\; \mathbf{q} \in \mathbb{R}^{768}
$$

$$
\text{"a landslide blocked the highway"} \;\xrightarrow{\;\text{encoder}\;}\; \mathbf{d} \in \mathbb{R}^{768}
$$

$$
\text{similarity} \;=\; \cos(\mathbf{q}, \mathbf{d}) \;=\; \frac{\mathbf{q}\cdot\mathbf{d}}{\lVert\mathbf{q}\rVert\,\lVert\mathbf{d}\rVert}
$$

So searching stops being about matching words. It becomes nearest neighbour
search in a vector space, and the query and the document never need to share a
single word.

Here is the whole pipeline. It is small enough that you can see all of it at
once.

| step | what happens | where it lives |
| --- | --- | --- |
| 1 | read 6 news stories from `data/retrieval/*.txt` | disk |
| 2 | split each story into sentences | a Python list |
| 3 | run each sentence through a frozen encoder | EmbeddingGemma |
| 4 | keep `{sentence_id: vector}` | a plain Python `dict` |
| 5 | encode the query the same way | the same encoder |
| 6 | cosine nearest neighbours, best sentences, best story | NumPy, 3 lines |

Two things are worth saying before we start.

1. **We train nothing.** There is not a single gradient step anywhere in this
   notebook. Every weight is downloaded, frozen and used exactly as it is.
   Somebody else paid for the training and we get a reusable map of meaning for
   free. That is the whole point of representation learning.
2. **The stories were written for this demo.** They follow the style of Nepali
   English language newspapers. The places are real, but the events, the numbers
   and the quotes are all invented. They are teaching texts and not reporting,
   so please do not quote them anywhere.

### 1. Setup

We need one extra package on top of our usual stack.

```bash
pip install "sentence-transformers>=5.0.0"
```

`sentence-transformers` is a thin wrapper around `transformers`. It takes care
of four things for us.

- It breaks the text into tokens.
- It runs the model in batches.
- It pools the token vectors into one vector per sentence.
- It normalises that vector to unit length.

We are going to use it as a black box. Today the encoder is a tool, and it is
not the thing we are studying.

In [ ]:
import re
import glob
import os

import numpy as np
import torch
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer

np.random.seed(0)
torch.manual_seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

### 2. The corpus: six stories

Each `.txt` file holds one story. The first line is the headline and the rest is
the body, with one paragraph per line.

Six documents is a ridiculous size for a search engine. That is exactly why it
is a good size for a classroom. You can read the whole collection in five
minutes, and after that you can judge every result yourself instead of trusting
the machine.

In [ ]:
DATA_DIR = "data/retrieval"

def load_stories(data_dir):
    stories = {}
    for path in sorted(glob.glob(os.path.join(data_dir, "*.txt"))):
        key = os.path.splitext(os.path.basename(path))[0]
        lines = [ln.strip() for ln in open(path, encoding="utf-8") if ln.strip()]
        stories[key] = {"title": lines[0], "paragraphs": lines[1:]}
    return stories

stories = load_stories(DATA_DIR)

print(f"{len(stories)} stories\n")
for key, s in stories.items():
    n_words = sum(len(p.split()) for p in s["paragraphs"])
    print(f"  {key:<24} {n_words:>4} words   {s['title']}")

In [ ]:
# what one story actually looks like
s = stories["01_landslide_highway"]
print(s["title"].upper(), "\n")
for p in s["paragraphs"][:3]:
    print(p, "\n")

### 3. Sentences are the unit of retrieval

We could make one vector for each story, but that would be a poor choice.

A story of 350 words is about many things at the same time. The landslide story
talks about rain, buses, excavators, a crack in the hillside and spoiling
vegetables. If we average all of that into a single vector, we get a blurry
point that sits close to nothing in particular.

So we embed sentences instead. A sentence is usually about one thing, and that
makes its vector sharp. We then build the story score out of the sentence
scores, which we do in section 8.

Real systems call this splitting step **chunking**. Choosing the chunk size is
one of the main design decisions you make when you build a retrieval system.

The splitter below breaks on `.`, `?` and `!`, and also on `।`, the Devanagari
danda. That way the same function keeps working if you drop Nepali script
articles into `data/retrieval/`.

In [ ]:
SENT_END = re.compile(r"(?<=[.!?।])\s+")

def split_sentences(text, min_chars=40):
    parts = [p.strip() for p in SENT_END.split(text)]
    return [p for p in parts if len(p) >= min_chars]

# flatten the corpus into one list of (story_key, sentence)
records = []
for key, s in stories.items():
    records.append((key, s["title"]))                     # the headline counts as a sentence
    for para in s["paragraphs"]:
        for sent in split_sentences(para):
            records.append((key, sent))

print(f"{len(records)} sentences from {len(stories)} stories\n")
for key, sent in records[:4]:
    print(f"[{key}] {sent}")

### 4. The encoder: EmbeddingGemma, frozen

[EmbeddingGemma 300M](https://huggingface.co/google/embeddinggemma-300m) is the
small embedding model from Google's Gemma family. Four facts matter for us.

- It has about 308 million parameters, so it runs comfortably on a CPU.
- Its output vector has 768 numbers.
- Its input window is 2048 tokens.
- It was trained on more than 100 languages, and Nepali is one of them.

We put that last point to use in section 10.

**You need to set up access once.** The model is gated, so you have to accept
the Gemma licence on the model page and then log in.

```bash
pip install -U huggingface_hub
hf auth login          # on older versions this was: huggingface-cli login
```

If you have not done that yet, the next cell falls back to an open multilingual
model so that the rest of the notebook still runs. Everything after that point
is identical, and that is a small lesson in itself, because the retrieval
machinery does not care which encoder produced the vectors.

**A word about the task prefixes.** EmbeddingGemma was trained with a short
instruction glued to the front of every text, and a query gets a different
prefix from a document. This is not decoration. It is how you tell the model
which of its behaviours you want, and using the wrong prefix makes retrieval
measurably worse.

In [ ]:
MODEL_CARDS = {
    "google/embeddinggemma-300m": {
        "query":      "task: search result | query: ",
        "document":   "title: none | text: ",
        "matryoshka": True,   # trained so that truncated vectors stay usable
    },
    "intfloat/multilingual-e5-small": {
        "query":      "query: ",
        "document":   "passage: ",
        "matryoshka": False,
    },
}

PREFERRED = "google/embeddinggemma-300m"
FALLBACK  = "intfloat/multilingual-e5-small"

try:
    MODEL_NAME = PREFERRED
    encoder = SentenceTransformer(MODEL_NAME, device=device)
except Exception as err:
    print(f"could not load {PREFERRED}:\n  {type(err).__name__}: {str(err)[:160]}")
    print(f"\nfalling back to the open model {FALLBACK}.")
    print("to use Gemma: accept the licence at huggingface.co/google/embeddinggemma-300m, then `hf auth login`.\n")
    MODEL_NAME = FALLBACK
    encoder = SentenceTransformer(MODEL_NAME, device=device)

CARD = MODEL_CARDS[MODEL_NAME]
print("encoder:", MODEL_NAME)

In [ ]:
n_params = sum(p.numel() for p in encoder.parameters())
n_train  = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
DIM = encoder.get_sentence_embedding_dimension()

print(f"parameters        : {n_params/1e6:,.1f} M")
print(f"embedding dim     : {DIM}")
print(f"max input tokens  : {encoder.max_seq_length}")
print(f"query prefix      : {CARD['query']!r}")
print(f"document prefix   : {CARD['document']!r}")

# freeze everything: this notebook never computes a gradient
encoder.eval()
for p in encoder.parameters():
    p.requires_grad_(False)
print(f"\ntrainable parameters after freezing: {sum(p.requires_grad for p in encoder.parameters())}")

### 5. What an embedding actually is

It is a list of numbers, and that is the whole object.

There are no words inside it and no grammar either. Nothing in it can be read by
a human. Its only useful property is **where it sits in relation to the other
embeddings**.

We ask the model for vectors of length 1, which places every sentence on the
surface of a unit sphere. Cosine similarity then becomes a plain dot product,
and the whole search turns into one matrix multiplication.

In [ ]:
def embed(texts, kind):
    # kind is either 'query' or 'document', and each one gets its own prefix
    prefixed = [CARD[kind] + t for t in texts]
    return encoder.encode(
        prefixed,
        normalize_embeddings=True,     # unit length, so cosine becomes a dot product
        batch_size=16,
        show_progress_bar=False,
    )

demo = "A landslide blocked the highway after heavy rain."
v = embed([demo], "document")[0]

print(f"text   : {demo}")
print(f"shape  : {v.shape}")
print(f"first 8: {np.round(v[:8], 4)}")
print(f"norm   : {np.linalg.norm(v):.4f}")

### 6. The index: a Python dictionary

The index is a plain `dict`. It maps a sentence id to the text, to the story it
came from, and to its vector. For 92 sentences that is perfectly adequate.

We also stack all the vectors into a single matrix of shape `(N, DIM)`, so we
end up with two views of the same thing.

- The dictionary is the part we **read**.
- The matrix is the part we **search**.

A production system replaces both of them with a vector database such as FAISS,
Qdrant or pgvector, which can run approximate nearest neighbour search over
millions of vectors. What those databases store is still exactly what you see
here.

In [ ]:
sent_texts = [sent for _, sent in records]

vectors = embed(sent_texts, "document")      # the only expensive step, and it runs once

INDEX = {
    i: {"story": key, "text": sent, "vector": vectors[i]}
    for i, (key, sent) in enumerate(records)
}

E = np.stack([INDEX[i]["vector"] for i in range(len(INDEX))])   # (N, DIM)

print(f"indexed sentences : {len(INDEX)}")
print(f"matrix E          : {E.shape}   ({E.nbytes/1024:.0f} KB)")
print()
print("INDEX[7] =", {"story": INDEX[7]["story"],
                     "text": INDEX[7]["text"][:60] + "...",
                     "vector": f"array of {DIM} floats"})

### 7. Nearest neighbour search

Here is the entire search engine.

$$
\mathbf{s} = E\,\mathbf{q}, \qquad E \in \mathbb{R}^{N \times d},\; \mathbf{q} \in \mathbb{R}^{d}
$$

Both sides are already normalised, so one matrix vector product gives us the
similarity between the query and every sentence in the corpus. We sort the
result and keep the top $k$. That is all there is to it, and those three lines
stay the same no matter how good the encoder becomes.

One warning before you read the numbers. The absolute value of a cosine score
belongs to the encoder and not to relevance. Some encoders squeeze every score
into the range 0.7 to 0.9, while others spread them from 0.1 to 0.8. Only two
things carry meaning.

- The order of the results.
- The gap between the first result and the ones behind it.

In [ ]:
def search_sentences(query, k=5):
    q = embed([query], "query")[0]      # note the query prefix, not the document one
    scores = E @ q                      # cosine similarity with every sentence
    top = np.argsort(-scores)[:k]
    return [(float(scores[i]), INDEX[i]["story"], INDEX[i]["text"]) for i in top]

def show_sentences(query, k=5):
    print(f"QUERY: {query}\n")
    for score, story, text in search_sentences(query, k):
        print(f"  {score:.3f}  [{story}]")
        print(f"         {text}\n")

show_sentences("Which road got buried by mud?")

Look carefully at what just happened. The query used the words *road*, *buried*
and *mud*. The winning sentence talks about a *landslide* on a *highway*. It won
because its vector sits near the query vector, and not because any word matched.

Here are two more queries. Both are phrased the way a student would ask them,
rather than the way the article is written.

In [ ]:
show_sentences("the rubbish people abandon on the peak", k=3)
print("-" * 70, "\n")
show_sentences("a nail-biting finish decided by the final delivery", k=3)

### 8. From sentences back to stories

The user asked for a story, but what we have is a score for every sentence. We
need a rule that turns one into the other, and the rule we choose changes the
answer. There are three sensible options.

- **The maximum.** A story scores as well as its single best sentence. This is
  very sensitive, because one good sentence is enough. It is also noisy, because
  one lucky sentence is equally enough.
- **The mean.** This averages over every sentence, including the nine that have
  nothing to do with the query, so longer stories get punished for being long.
- **The mean of the best $m$ sentences.** This is the compromise we use below,
  with $m$ set to 3. Now a story wins by being relevant more than once, and that
  is the behaviour we actually want.

In [ ]:
def search_stories(query, k=3, m=3):
    # rank each story by the mean of its m best sentence scores
    q = embed([query], "query")[0]
    scores = E @ q

    per_story = {}
    for i, sc in enumerate(scores):
        per_story.setdefault(INDEX[i]["story"], []).append((float(sc), INDEX[i]["text"]))

    ranked = []
    for key, hits in per_story.items():
        hits.sort(reverse=True)
        ranked.append((float(np.mean([h[0] for h in hits[:m]])), key, hits[0]))
    ranked.sort(reverse=True)
    return ranked[:k]

def answer(query, k=3):
    print(f"QUERY: {query}\n")
    for rank, (score, key, (best_sc, best_text)) in enumerate(search_stories(query, k), 1):
        mark = "-->" if rank == 1 else "   "
        print(f"{mark} {score:.3f}  {stories[key]['title']}")
        print(f"          best sentence ({best_sc:.3f}): {best_text[:100]}...\n")

answer("why do people complain about too many climbing permits")

In [ ]:
for q in [
    "the dam is holding water for the dry season",
    "where do migrant workers send their money",
    "an aerodrome with hardly any flights",
]:
    answer(q, k=2)
    print("=" * 78, "\n")

### 9. The comparison that makes the point: keyword search

Now we run the control experiment.

TF-IDF is a strong and honest keyword baseline. It scores a document by the
words it shares with the query, and it gives rare words more weight than common
ones. It was the backbone of search for decades, so it deserves a fair test.

Its ceiling is a hard one though. If the query and the document share no words
at all, the score is exactly zero. No amount of clever weighting can get around
that, because the representation itself is nothing more than a bag of word
counts.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words="english").fit(sent_texts)
T = tfidf.transform(sent_texts)            # sparse (N, vocab)

def keyword_top(query):
    qv = tfidf.transform([query])
    sc = (T @ qv.T).toarray().ravel()
    i = int(np.argmax(sc))
    return float(sc[i]), INDEX[i]["story"], INDEX[i]["text"]

probes = [
    "vehicles stuck because of debris on the hill road",
    "the rubbish people abandon on the peak",
    "a nail-biting finish decided by the final delivery",
    "an aerodrome with hardly any flights",
    "what is the price of gold today?",          # the corpus cannot answer this one
]

for q in probes:
    k_sc, k_story, k_text = keyword_top(q)
    e_sc, e_story, e_text = search_sentences(q, k=1)[0]
    if k_sc == 0.0:
        k_story, k_text = "no word in common", ""
    print(f"QUERY   : {q}")
    print(f"  tfidf   {k_sc:.3f}  [{k_story:<20}] {k_text[:58]}")
    print(f"  embed   {e_sc:.3f}  [{e_story:<20}] {e_text[:58]}")
    print()

There are three things to look at in that output.

1. **The scores.** TF-IDF lands somewhere around 0.2 or 0.3 on a paraphrased
   query, while the embedding stays high. A score that low is barely better than
   noise. On a real corpus of a million documents, dozens of irrelevant ones
   would beat it by pure accident.
2. **The wrong answers.** At least one query sends TF-IDF to the wrong story
   completely, because it latched on to one word that the two texts happen to
   share. The embedding gets the same query right.
3. **The zero.** The gold price query shares no word with anything in the
   corpus, so TF-IDF scores exactly 0 and has nothing to rank at all. That is
   the hard ceiling we just talked about, because a bag of word counts cannot
   represent a meaning it has no word for.

Now read that last line once more. The embedding also answers the gold price
query, its score looks confident, and its answer is nonsense. We will come back
to that.

None of this makes keyword search useless. It is precise on names, codes and
exact phrases, which is where embeddings are vague. That is why serious systems
run both of them and then merge the two rankings, an approach they call hybrid
search.

### 10. The part that surprises students: cross lingual retrieval

The whole corpus is in English. The queries below are in Nepali. Nobody
translated anything, and no Nepali text was ever added to the index.

A multilingual encoder is trained so that a sentence and its translation land at
almost the same point. Meaning and language end up stored separately, so the
position tells you what is being said, while the language it was said in is
largely factored out.

A Nepali query therefore lands in the neighbourhood of the English sentences
that mean the same thing, and nearest neighbour search works without a single
change.

In [ ]:
nepali_queries = [
    "पहिरोले सडक बन्द भयो",                    # a landslide closed the road
    "विदेशबाट पठाएको पैसा",                     # money sent from abroad
    "नेपालले बिजुली बेच्यो",                      # Nepal sold electricity
    "हिमाल सफाइ अभियान",                       # mountain cleanup campaign
]

for q in nepali_queries:
    score, key, (best_sc, best_text) = search_stories(q, k=1)[0]
    print(f"{q}")
    print(f"   --> {score:.3f}  {stories[key]['title']}")
    print(f"       {best_text[:90]}...\n")

### 11. A map of the embedding space

We cannot draw several hundred dimensions, so we squash them down to 2 with
t-SNE.

**Measure first and look second.** A 2D projection is an illustration and never
evidence. t-SNE keeps track of which points are neighbours and throws away
everything else, so in the picture below you should ignore three things.

- The size of a blob means nothing.
- The width of a gap means nothing.
- The distance between two clusters means nothing.

Because of that we first measure the claim in the full space, by asking how
often a sentence's nearest neighbours come from its own story. Only then do we
draw the picture.

Keep in mind that nobody ever told the encoder which story a sentence came from.
The grouping appears on its own, simply because those sentences mean related
things.

The query is projected together with the sentences and drawn as a star, and
dashed lines join it to its three real nearest neighbours. Retrieval is
literally the question "what is near the star".

In [ ]:
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors

query = "vehicles stuck because of debris on the hill road"
qv = embed([query], "query")[0]

# first the evidence, measured in the full space
labels = [INDEX[i]["story"] for i in range(len(INDEX))]
_, nbr = NearestNeighbors(n_neighbors=6, metric="cosine").fit(E).kneighbors(E)
purity = np.mean([[labels[j] == labels[i] for j in nbr[i][1:]] for i in range(len(INDEX))])
print(f"a sentence's 5 nearest neighbours come from its own story {purity:.0%} of the time")
print(f"(if the space carried no meaning, this would be about {1/len(stories):.0%})")

# then the illustration, projecting the sentences and the query together
P = TSNE(n_components=2, metric="cosine", init="pca",
         perplexity=12, random_state=0).fit_transform(np.vstack([E, qv[None, :]]))
pq, P = P[-1], P[:-1]

keys = list(stories.keys())
colors = plt.cm.tab10(np.linspace(0, 1, len(keys)))

fig, ax = plt.subplots(figsize=(9, 6))
for c, key in zip(colors, keys):
    mask = np.array([lab == key for lab in labels])
    ax.scatter(P[mask, 0], P[mask, 1], color=c, s=42, alpha=0.8,
               label=stories[key]["title"][:40])

ax.scatter(*pq, marker="*", s=620, color="black", zorder=5, label="QUERY")
for i in np.argsort(-(E @ qv))[:3]:                 # neighbours come from the full space
    ax.plot([pq[0], P[i, 0]], [pq[1], P[i, 1]], "k--", lw=0.9, alpha=0.6, zorder=4)

ax.set_xticks([]); ax.set_yticks([])
ax.set_title("sentence embeddings, t-SNE to 2D (axes have no meaning)")
ax.legend(fontsize=8, loc="center left", bbox_to_anchor=(1.01, 0.5))
plt.tight_layout(); plt.show()

This is the same structure once more, but now without any projection.

We sort the sentences by story and plot every pairwise cosine similarity. The
bright blocks along the diagonal are the clusters, and this time we are looking
at them in the real space instead of a flattened shadow of it.

The warm patches away from the diagonal are not mistakes. They are genuine
overlap in meaning.

- Mountains connect the Everest story to the tourism story.
- Rivers and the monsoon connect the landslide story to the hydropower story.

A keyword index has no way of saying "these two stories are somewhat related".
This one says it as a number, and we got that for free.

In [ ]:
order = np.argsort([keys.index(INDEX[i]["story"]) for i in range(len(INDEX))])
S = E[order] @ E[order].T
np.fill_diagonal(S, np.nan)          # a sentence matches itself at 1.0 and would eat the colour range

fig, ax = plt.subplots(figsize=(7.5, 6.2))
im = ax.imshow(S, cmap="magma",
               vmin=np.nanpercentile(S, 2), vmax=np.nanpercentile(S, 98))

bounds, pos = [], 0
for key in keys:
    n = sum(1 for i in range(len(INDEX)) if INDEX[i]["story"] == key)
    pos += n; bounds.append(pos)
for b in bounds[:-1]:
    ax.axhline(b - 0.5, color="white", lw=0.8)
    ax.axvline(b - 0.5, color="white", lw=0.8)

centers = [(([0] + bounds)[i] + bounds[i]) / 2 for i in range(len(keys))]
ax.set_xticks(centers); ax.set_xticklabels([k[:2] for k in keys])
ax.set_yticks(centers); ax.set_yticklabels([k[3:18] for k in keys], fontsize=8)
ax.set_title("pairwise cosine similarity, sentences grouped by story")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.show()

### 12. Bonus: how small can the vectors get?

EmbeddingGemma is trained with something called Matryoshka representation
learning. The information is packed so that the first $d$ numbers of a vector
are themselves a valid, smaller embedding.

That means you can cut a 768 number vector down to 128, normalise it again, and
keep most of the retrieval quality while using one sixth of the memory. At a
billion documents that is the difference between one machine and six.

We measure it by asking how often the shortened index still returns the same top
sentence as the full width index. Watch the sentence column, because that is the
sensitive one. The story column is much more forgiving, since with only six
stories a wrong sentence often still sits inside the right story.

If you are running the fallback encoder, expect the sentence column to fall
apart below half width. An ordinary encoder spreads its information across all
of its dimensions, so cutting them off destroys it. That contrast is the real
lesson here, because the free lunch comes from Matryoshka training and not from
truncation by itself.

In [ ]:
def truncate(M, d):
    Md = M[:, :d]
    return Md / np.linalg.norm(Md, axis=1, keepdims=True)

test_queries = probes + [
    "why do people complain about too many climbing permits",
    "where do migrant workers send their money",
    "the dam is holding water for the dry season",
    "how do people pay their bills from a phone",
    "visitors are staying for a shorter time",
]
Q = np.stack([embed([q], "query")[0] for q in test_queries])

full_sent = np.argmax(Q @ E.T, axis=1)                        # the best sentence at full width
full_story = [INDEX[i]["story"] for i in full_sent]

print(f"encoder: {MODEL_NAME}   (matryoshka-trained: {CARD['matryoshka']})")
print(f"{len(test_queries)} test queries, {len(INDEX)} sentences\n")
print(f"{'dims':>5} {'index size':>11} {'same sentence':>14} {'same story':>11}")
for d in [768, 512, 384, 256, 128, 64, 32]:
    if d > DIM:
        continue
    Ed, Qd = truncate(E, d), truncate(Q, d)
    best = np.argmax(Qd @ Ed.T, axis=1)
    same_sent = np.mean(best == full_sent)
    same_story = np.mean([INDEX[i]["story"] == s for i, s in zip(best, full_story)])
    print(f"{d:>5} {Ed.nbytes/1024:>10.0f}K {same_sent:>13.0%} {same_story:>10.0%}")

### 13. Your turn: ask it something

Run the cell and type a question in English or in Nepali. Press enter on an
empty line to stop.

Three things are worth trying in front of the class.

1. A paraphrase that shares no word with the article, such as "who is cleaning
   up the rubbish up high?"
2. A question the corpus genuinely cannot answer, such as "what is the price of
   gold today?" Watch it return something anyway and look confident about it,
   because nearest neighbour search always has a nearest neighbour.
3. The same question asked once in Nepali and once in English, so that you can
   compare the two scores.

In [ ]:
while True:
    q = input("\nquery (blank to stop): ").strip()
    if not q:
        print("done.")
        break
    print()
    answer(q, k=3)

### What to take away

**1. The encoder stayed frozen.** We trained nothing at all. Representation
learning means that the expensive part, which is turning language into geometry,
was done once by somebody else and now works as a reusable component. The same
frozen vectors would also serve classification, clustering or duplicate
detection without any change.

**2. Retrieval is geometry.** Once a text is a point on a sphere, "find the
relevant document" becomes "find the nearest point", and that is one matrix
multiplication. All of the difficulty moved into the encoder, so the search
itself is first year linear algebra.

**3. Meaning beats matching.** The TF-IDF table in section 9 is the whole
argument for embeddings in one place. The Nepali queries in section 10 show how
far the idea reaches, because the representation is about meaning and the
language turns out to be almost incidental to it.

**4. It always gives you an answer.** Nearest neighbour search cannot say "I do
not know", because there is always a nearest vector. Real systems add a
similarity threshold and refuse to answer below it. Try the gold price query
again and look at the score it gets.

**5. This is the R in RAG.** Retrieval augmented generation is this notebook
plus one more step. You take the top scoring text, paste it into an LLM prompt,
and ask the model to answer using only that text. Everything that makes RAG work
or fail happens in the part we have just built.

### Exercises

1. **Chunk size.** Index whole paragraphs instead of sentences, and then try
   whole stories. Where does retrieval get better, and where does it go blurry?
2. **The prefixes.** Encode the queries with the document prefix by mistake and
   run section 7 again. How much does the ranking suffer?
3. **Aggregation.** Set `m` in `search_stories` to 1, which gives you the
   maximum, and then to 50, which is effectively the mean. Find a query where
   the two rules disagree.
4. **A threshold.** Print "no relevant story found" when the top score falls
   below some value, say 0.55. Use questions the corpus cannot answer to
   calibrate it.
5. **Your own corpus.** Put five articles in Nepali script into
   `data/retrieval/` and run the notebook again. Nothing in the code needs to
   change.